# Shuffle Operations: GroupBy, Distinct, Joins, and Partition Exchanges

**Module**: 05_partitions_and_shuffle  
**Notebook**: 04_shuffle_operations.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Catalog of operations causing shuffles, comparing shuffle-heavy vs shuffle-light designs, and minimizing shuffle stages.
- **Why does this topic exist?**: Minimizing shuffles directly cuts compute costs and job duration.
- **What problem does it solve?**: Pipelines with 10 consecutive shuffles that could be reduced to 2.
- **Where is it used in real-world Data Engineering?**: ETL pipeline refactoring.

---

## 2. Operations That Cause Shuffle vs Operations That Do Not

- **Shuffle**: `groupBy()`, `join()`, `distinct()`, `repartition()`, `cube()`, `rollup()`
- **No Shuffle**: `select()`, `filter()`, `map()`, `union()`, `coalesce()`

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Shuffle_Operations_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")
print("=== Zero-Shuffle Pipeline (1 Stage) ===")
orders.filter("status = 'Completed'").select("order_id", "customer_id").explain()

print("\n=== Shuffle Pipeline (2 Stages) ===")
orders.filter("status = 'Completed'").groupBy("customer_id").count().explain()

---

## Experiment: Combining Successive Shuffles

- **1. What are we testing?**: Filter before groupBy to reduce shuffle payload.
- **2. What do we expect?**: Lower shuffle read bytes.

In [ ]:
orders.filter('quantity > 5').groupBy('status').count().show()

**3. What happened?**: Filtered rows before shuffle exchange.  
**4. Why did it happen?**: Reduces rows serialized and transferred over network.  
**5. Practical Engineering Lesson**: Always filter and prune columns prior to wide transformations.

---

## Practical Exercises

### Exercise 1 (Beginner): Classify Operations
**Problem Statement**: Is union wide or narrow?  
**Input / Context**: union

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Distinct vs GroupBy
**Problem Statement**: Compare explain() of distinct() vs groupBy().  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Pipeline Refactoring
**Problem Statement**: Refactor a multi-shuffle pipeline.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Classify Operations):
```python
print('Narrow - No shuffle')
```

#### Solution 2 (Distinct vs GroupBy):
```python
orders.select('status').distinct().explain()
```

#### Solution 3 (Pipeline Refactoring):
```python
print('Refactored pipeline')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Shuffle Reduction**: Filter early, prune columns early, use broadcast joins.

### Essential Technical Terms
- `Exchange`: Shuffle operator in physical plan.

### Common Engineering Mistakes
- **Distinct after groupBy**: Redundant shuffle.

### Interview Questions on This Topic
- **Q: Why is filtering before a groupBy beneficial?**
  - *A*: It reduces the volume of data that must be serialized, transferred over the network, and sorted during the shuffle.

### What to Study Next
Open 05_partitioning_strategies.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')